Benotet Option:
Variant 4. Personal finance analyzer
The system must:
 record financial transactions
 categorize expenses
 summarize spending by category
 provide simple insights (e.g., highest spending category)
The system must support multiple transactions and generate meaningful summaries.

Develop class structure 
Data:
transation date, card number, description, category, amount, unique number

Expense_Manager
    add, delete, update, calculations, insights
Expense
    attributes: private number
    transation date, description, category, amount
Report


In [ ]:
# models.py Classes (e.g. Task, Item, Record)
class Expense:
    # primary expense class to hold the data for each expense entry
    def __init__(self, number, description, category, amount):
        self._number = number
        self.description = description
        self.category = category
        self.amount = float(amount)

# set up based category rules for the expense manager to use when categorizing expenses
# data structure: dictionary of sets to avoid duplicates and allow for faster lookups
category_rules = {
    "Groceries": {"grocery", "supermarket", "food"},
    "Housing": {"rent", "mortgage"},
    "Utilities": {"electricity", "water", "gas"},
    "Dining Out": {"restaurant", "dining", "cafe"},
    "Transportation": {"transportation", "bus", "train", "taxi"}
}

# logic.py ? This section outlines the business operations and primary functions
class Expense_Manager:
    def __init__(self):
        self.expenses = {}
        self._next_number = 1
        self._category_rules = dict(category_rules)
        
    def add_expense(self, description, amount, category=None):
        if category is None:
            category = self.categorize_expense(description)
        else:
            if category not in self._category_rules:
                self._category_rules[category] = []
        expense = Expense(self._next_number, description, category, amount)
        self.expenses[self._next_number] = expense
        self._next_number += 1
        return expense

    def delete_expense(self, number):
        if number in self.expenses:
            del self.expenses[number]
            return True
        return False

    def update_expense(self, number, new_description=None, new_category=None, new_amount=None):
        if number in self.expenses:
            expense = self.expenses[number]
            if new_description is not None:
                expense.description = new_description
            if new_category is not None:
                expense.category = new_category
            if new_amount is not None:
                    expense.amount = float(new_amount)
            return True
        return False

    def display_expenses(self):
        # unordered list of all expenses with their details
        for expense in self.expenses.values():
            print(expense._number, expense.description, "-", expense.category, "-", expense.amount)

    def categorize_expense(self, description: str) -> str:
        # checks the description against the category rules and returns the appropriate category
        desc_lower = description.lower()
        for category, keywords in self._category_rules.items():
            if any(keyword in desc_lower for keyword in keywords):
                return category
        else:
            return "Other"
        
    def add_category_rule(self, category: str, keywords: list) -> None:
        # checks if the category already exists, if so, adds the keywords to the existing set, 
        # otherwise creates a new set for the category
        low_keywords = {kw.lower() for kw in keywords}
        if category in self._category_rules:
            self._category_rules[category].update(low_keywords)
        else:
            self._category_rules[category] = (low_keywords)
        
    def summarize_by_category(self):
        # returns a dictionary with the total amount spent in each category
        # key: category name, value: total amount spent
        summary = {category: 0 for category in self._category_rules.keys()}
        summary["Other"] = 0
        for expense in self.expenses.values():
                # loop thorugh recorded expenses and add the amount to the right category in the summary
                summary[expense.category] += expense.amount
        return summary

    def highest_spending_category(self):
        # finds the category with the highest total spending and returns it
        # spentbycategoryreport to use this function to display the highest spending category
        summary = self.summarize_by_category()
        if not summary:
            return None
        return max(summary, key=summary.get)


Following section creates a reporting structure class based of of reporting structure in lecture (OOP 3)

In [ ]:
# models.py Classes
from typing import List, Optional
class ExpenseReporter:
    # Class-level Constants (Defaults)
    # EXPENSE_LIST_LENGTH: int = 40
    # AMOUNT_LENGTH: int = 10  
    # COL_NAMES: List[str] = ["DESCRIPTION", "AMOUNT"]
    # COL_AMOUNT: str = "AMOUNT"
    HEADER_SPLITER_CHARACTER = '_'
    LINE_SPLITER_CHARACTER = '-'

    def __init__(
        self,
        expenses: list,
        col_names: List[str],
        col_lengths: List[str],
        header_spliter_character: Optional[str] = HEADER_SPLITER_CHARACTER,
        line_spliter_character: Optional[str] = LINE_SPLITER_CHARACTER
    ) -> None:
        self.expenses = expenses
        self.col_names = col_names
        self.col_lengths = col_lengths
        self.header_spliter_character = header_spliter_character
        self.line_spliter_character = line_spliter_character
        self.total_length = sum(self.col_lengths)

    # --- Header Component Methods ---
    def _get_header_spliter(self) -> str:
        return self.header_spliter_character * self.total_length

    def _get_header(self) -> str:
        # centers every column name in its specified width
        return "".join(f"{name:^{length}}" for name, length in zip(self.col_names, self.col_lengths))

    def _output_header(self) -> None:
        header = self._get_header()
        print(header)
        spliter = self._get_header_spliter()
        print(spliter)

    # --- List Item Component Methods ---
    def _handle_data_name(self, text: str, max_width: int) -> str:
        text_str = str(text)
        if len(text_str) > max_width:
            half_text_width = int((max_width / 2) - 1)
            left_text = text[:half_text_width]
            right_text = text[-half_text_width:]
            return f'{left_text}..{right_text}'
        return text_str

    def _get_line_spliter(self) -> str:
        size = self.total_length
        return self.line_spliter_character * size
    
    def _get_line_item(self, row_data: List[str]) -> str:
        row_str = ""
        for i, data in enumerate(row_data):
            length = self.col_lengths[i] 
            safe_data = self._handle_data_name(data, length)
            if i < len(self.col_lengths) - 1:  # For all but the last column, left-align
                row_str += f"{safe_data:<{length}}  "
            else:
                row_str += f"{safe_data:>{length}}"
        return row_str

    def _output_line_item(self, row_data: List[str]) -> None:
        line_item = self._get_line_item(row_data)
        print(line_item)
        spliter = self._get_line_spliter()
        print(spliter)

    # --- Summary Component Methods ---
    def _get_total_amount(self) -> float:
        return sum([expense.amount for expense in self.expenses.values()])

    def _get_summerize(self, total_amount: float) -> str:
        return f'Total: {total_amount:.2f}'

    def _get_custom_insights(self) -> str:
        # Placeholder for child classes to override with specific insights
        return ""

    def _output_summary(self) -> None:
        total_amount = self._get_total_amount()
        summary_str = self._get_summerize(total_amount)
        print(f"{summary_str:>{self.total_length}}")
        insights = self._get_custom_insights()
        if insights:
            print(insights)

    # --- Public Main Execution Method ---
    def report(self) -> None:
        self._output_header()
        self._output_list_item()
        self._output_summary()



In [ ]:
# AllDataReport subclass of ExpenseReporter to display all data in a table format
class AllDataReport (ExpenseReporter):
    # Inheritance from ExpenseReporter
    def __init__(self, expenses: dict):
        columns = ["ID","Description", "Category", "Amount"]
        widths = [5, 30, 20, 15]
        super().__init__(expenses=expenses, col_names=columns, col_lengths=widths)
            #passes configuration up to the parent class constructor for column names and widths

    def _output_list_item(self) -> None:
        # updates output_list_item to display all data
        for expense in self.expenses.values():
            # loops through expenses and prepares the row data for each expense 
            row_data = [
                str(expense._number),
                expense.description,
                expense.category,
                f"{expense.amount:.2f}" #forces string formatting with 2 decimal places
            ]
            self._output_line_item(row_data)

In [ ]:
# SpentbyCategoryReport subclass of ExpenseReporter to display spending by category in a table format
class SpentbyCategoryReport (ExpenseReporter):
    # Inheritance from ExpenseReporter
    def __init__(self, manager: Expense_Manager):
        columns = ["Category", "Amount"]
        widths = [30, 15]
        super().__init__(expenses=manager.expenses, col_names=columns, col_lengths=widths)
        self.manager = manager

    def _output_list_item(self) -> None:
        category_totals = self.manager.summarize_by_category()
        #updated to sort the categories by amount in descending order for better readability
        # 2. SORTING LOGIC: Convert the dictionary items to a list and sort by value
        # By calling category_totals.items(), you extract the calculation results into a list of tuples:
        # sorted() takes an iterable and returns a new sorted list (use dictionaries to store and calculate, lists to display data)
        # key=lambda item: item[1] tells Python to sort by the dictionary value (the amount)
        # reverse=True forces the order to be descending (Largest -> Smallest)
        sorted_categories = sorted(category_totals.items(), key=lambda item: item[1], reverse=True)  # Sort by amount (descending)
        
        for category, amount in sorted_categories:
            row_data = [
                category,
                f"{amount:.2f}"
            ]
            self._output_line_item(row_data)

In [ ]:
# Test space of all functions (Chat generated data)
if __name__ == "__main__":
    # Initialize the centralized data logging instance
    manager = Expense_Manager()
    
    print("=== STEP 1: LOGGING TRANSACTIONS WITH EXPLICIT CUSTOM CATEGORIES ===")
    # Testing dynamic category registration logic for custom, user-defined labels
    manager.add_expense("Grocery Store: Pennys", 45.05, "Food")
    manager.add_expense("Sushi Bar Dinner that cost too much", 65.00, "Food")
    manager.add_expense("DB Train Ticket", 20.85, "Transport")
    manager.add_expense("Monthly Bus Pass", 50.00, "Transport")
    manager.add_expense("Gym Membership", 30.00, "Health")
    print(f"Total entries logged: {len(manager.expenses)}")
    print("-" * 50)

    print("\n=== STEP 2: TESTING KEYWORD AUTO-CATEGORIZATION (FALLBACKS & MATCHES) ===")
    # Test 2A: Should match standard default rules ("supermarket" -> "Groceries")
    exp_auto = manager.add_expense("Weekly run to the local supermarket", 82.30)
    print(f"Auto-Matched: '{exp_auto.description}' -> Assigned: {exp_auto.category} (Expected: Groceries)")
    
    # Test 2B: Unrecognized terms should route securely into the "Other" safety bucket
    exp_fallback = manager.add_expense("Steam Game Purchase: Elden Ring", 59.99)
    print(f"Fallback Catch: '{exp_fallback.description}' -> Assigned: {exp_fallback.category} (Expected: Other)")
    print("-" * 50)

    print("\n=== STEP 3: TESTING DYNAMIC RULE INJECTION & LEARNING ===")
    # Registering a completely new category to the hash-map via user rules
    manager.add_category_rule("Entertainment", ["steam", "game", "cinema", "netflix"])
    # Add new store keywords to the existing "Groceries" rule
    manager.add_category_rule("Groceries", ["Lidl", "Rewe", "grocery"]) # 'grocery' is a duplicate!

    # Auto-log an expense using one of the newly injected keywords
    new_exp = manager.add_expense("Midnight snack run to Lidl", 14.20)
    print(f"Result: '{new_exp.description}' -> Categorized as: {new_exp.category} (Expected: Groceries)")
    # Output: Categorized as: Groceries
    
    # Adding a matching expense AFTER the rule is introduced to verify runtime expansion
    exp_learned = manager.add_expense("Steam Game Purchase: Cyberpunk", 49.99)
    print(f"Post-Rule Execution: '{exp_learned.description}' -> Assigned: {exp_learned.category} (Expected: Entertainment)")
    print("-" * 50)

    print("\n=== STEP 4: TROUBLESHOOTING UPDATES & MUTATION LOGIC ===")
    # Grab the transaction we want to manipulate (ID 1: Grocery Store: Pennys)
    target_id = 1
    print(f"Original Record (ID {target_id}): {manager.expenses[target_id].description} | Amount: €{manager.expenses[target_id].amount}")
    
    # Modify details and recalculate values
    manager.update_expense(target_id, new_description="Pennys Grocery Run (Refund Adjusted)", new_amount=35.00)
    print(f"Mutated Record (ID {target_id}): {manager.expenses[target_id].description} | Amount: €{manager.expenses[target_id].amount} (Expected: €35.00)")
    print("-" * 50)

    print("\n=== STEP 5: TROUBLESHOOTING DELETION LOGIC & ID STABILITY ===")
    # Delete an item from the middle of the stack (ID 3: DB Train Ticket)
    delete_target = 3
    success = manager.delete_expense(delete_target)
    print(f"Deletion Request for ID {delete_target}: {'SUCCESSFUL' if success else 'FAILED'}")
    print(f"Is ID {delete_target} still in active records? {delete_target in manager.expenses} (Expected: False)")
    print("-" * 50)

    print("\n=== STEP 6: VERIFYING MASTER SYSTEM CATEGORY BLUEPRINT ===")
    # Diagnostic layout loop to display the state of all internal tracking rules
    for category, keywords in manager._category_rules.items():
        print(f"• {category}: {keywords}")
    print("-" * 50)

    print("\n=== STEP 7: COMPILING INSIGHT METRICS ===")
    # Pull dynamic insights directly out of the business logic core
    highest = manager.highest_spending_category()
    print(f"Financial Insights Summary -> Highest Spending Category: {highest}")
    print("-" * 50)

    print("\n=== STEP 8: GENERATING TRANSACTIONAL DATA REPORT ===")
    # Compiles full analytical breakdown table
    data_reporter = AllDataReport(manager.expenses)
    data_reporter.report()

    print("\n=== STEP 9: GENERATING SPENDING BY CATEGORY SUMMARY REPORT ===")
    # Verifies complete, zero-balance accounting across all structural labels
    cat_reporter = SpentbyCategoryReport(manager)
    cat_reporter.report()

=== STEP 1: LOGGING TRANSACTIONS WITH EXPLICIT CUSTOM CATEGORIES ===
Total entries logged: 5
--------------------------------------------------

=== STEP 2: TESTING KEYWORD AUTO-CATEGORIZATION (FALLBACKS & MATCHES) ===
Auto-Matched: 'Weekly run to the local supermarket' -> Assigned: Groceries (Expected: Groceries)
Fallback Catch: 'Steam Game Purchase: Elden Ring' -> Assigned: Other (Expected: Other)
--------------------------------------------------

=== STEP 3: TESTING DYNAMIC RULE INJECTION & LEARNING ===
Result: 'Midnight snack run to Lidl' -> Categorized as: Groceries (Expected: Groceries)
Post-Rule Execution: 'Steam Game Purchase: Cyberpunk' -> Assigned: Entertainment (Expected: Entertainment)
--------------------------------------------------

=== STEP 4: TROUBLESHOOTING UPDATES & MUTATION LOGIC ===
Original Record (ID 1): Grocery Store: Pennys | Amount: €45.05
Mutated Record (ID 1): Pennys Grocery Run (Refund Adjusted) | Amount: €35.0 (Expected: €35.00)
-------------------------

26.06.2026, reviewed the data structures and updated category kewords to sets, converted report data to lists for deplay, troubleshot category keyword additions and user additions. 
    considered a graph function to use multiple accounts. where the transactions would move through the vertices and edges, such as transfering funds between checking and savings, and paying credit accounts. This would supoprt total net worth functionality and make sure that transaftsion such as paying off the credit car with separate expenses from a debit aren't repeated. Or paying yourself to savings. this is an optional addition to be considered. Another option is the data storage so that changes made are kept even when closing the program. 
Next step: helper functions to ensure that data entry is correct and clear 
Next step: Terminal Based Interface
 Your application must include a clear terminal-based interface that allows users to:
o input data
o trigger actions
o receive meaningful output